<a href="https://colab.research.google.com/github/yiyu-chen-labs/llm-from-scratch/blob/main/Week11_day1_encoders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U sentence-transformers

import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
bi = SentenceTransformer("BAAI/bge-m3", device=device)               # bi-encoder
cross = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device)       # cross-encoder

cases = {
    "有給休暇の繰越上限は何日？": [
        "有給休暇は翌年度に最大20日まで繰り越せる。",
        "2026年度より有給休暇の取得ルールを変更し、特別休暇を新設した。",
        "社員食堂の営業時間は11時から14時までです。",
    ],
    "退職金が出ない条件は？": [
        "勤続3年未満で自己都合退職した場合、退職金は支給されない。",
        "勤続3年以上の社員には退職金が支給される。",
        "通勤手当は月額3万円を上限として支給する。",
    ],
}

for query, docs in cases.items():
    q = bi.encode(query, normalize_embeddings=True)       # (1024,)
    D = bi.encode(docs, normalize_embeddings=True)        # (3, 1024)
    bi_scores = D @ q                                      # 正規化後，點積 = cosine
    cross_scores = cross.predict([(query, d) for d in docs])  # 每一對 (query, 文件) 一個分數

    print(f"\nQ: {query}")
    for d, b, c in zip(docs, bi_scores, cross_scores):
        print(f"  bi={b:.3f}  cross={c:.3f}  {d}")



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


Q: 有給休暇の繰越上限は何日？
  bi=0.749  cross=0.993  有給休暇は翌年度に最大20日まで繰り越せる。
  bi=0.573  cross=0.001  2026年度より有給休暇の取得ルールを変更し、特別休暇を新設した。
  bi=0.417  cross=0.000  社員食堂の営業時間は11時から14時までです。

Q: 退職金が出ない条件は？
  bi=0.737  cross=0.707  勤続3年未満で自己都合退職した場合、退職金は支給されない。
  bi=0.590  cross=0.014  勤続3年以上の社員には退職金が支給される。
  bi=0.462  cross=0.000  通勤手当は月額3万円を上限として支給する。


In [3]:
# 2. 一個問題，三份文件
query = "有給休暇の繰越上限は何日？"

docs = [
    "有給休暇は翌年度に最大20日まで繰り越せる。",                       # 正確答案
    "2026年度より有給休暇の取得ルールを変更し、特別休暇を新設した。",   # 很像但不對
    "社員食堂の営業時間は11時から14時までです。",                       # 完全無關
]


In [9]:
# 3. 自己寫點積：兩個向量對應位置相乘，全部加起來
def dot(a, b):
  total = 0
  for i in range(len(a)):
    total += a[i]*b[i]
  return total


In [13]:
# 4. Bi-encoder：問題和文件各自變成向量，再比較
print("=== Bi-encoder ===")
q_vec = bi.encode(query, normalize_embeddings=True)
print("問題向量的長度：", len(q_vec))


for doc in docs:
  d_vec = bi.encode(doc, normalize_embeddings=True)
  score = dot(q_vec, d_vec)
  print(f"bi={score:.3f}  {doc}")
  # print("文件向量的長度：", len(d_vec))
  print("---------------------------------")

=== Bi-encoder ===
問題向量的長度： 1024
bi=0.749  有給休暇は翌年度に最大20日まで繰り越せる。
---------------------------------
bi=0.573  2026年度より有給休暇の取得ルールを変更し、特別休暇を新設した。
---------------------------------
bi=0.417  社員食堂の営業時間は11時から14時までです。
---------------------------------


In [14]:
# 5. Cross-encoder：問題和文件放在一起，直接給分數

print("=== cross-encoded ===")
for doc in docs:
  pair = (query, doc)
  score = cross.predict(pair)
  print(f"cross={score:.3f}  {doc}")


=== cross-encoded ===
cross=0.993  有給休暇は翌年度に最大20日まで繰り越せる。
cross=0.001  2026年度より有給休暇の取得ルールを変更し、特別休暇を新設した。
cross=0.000  社員食堂の営業時間は11時から14時までです。


In [15]:
#----case 2 ------#

In [24]:
query = "勤続3年未満だと退職金は出ない？"
docs = [
    "勤続3年未満の場合、退職金は支給されない。",
    "勤続3年未満の場合、退職金は支給される。",
    "通勤手当は月額3万円を上限として支給する。",
]

In [25]:
# 3. 自己寫點積：兩個向量對應位置相乘，全部加起來
def dot(a, b):
  total = 0
  for i in range(len(a)):
    total += a[i]*b[i]
  return total


In [26]:
# 4. Bi-encoder：問題和文件各自變成向量，再比較
print("=== Bi-encoder ===")
q_vec = bi.encode(query, normalize_embeddings=True)
print("問題向量的長度：", len(q_vec))


for doc in docs:
  d_vec = bi.encode(doc, normalize_embeddings=True)
  score = dot(q_vec, d_vec)
  print(f"bi={score:.3f}  {doc}")
  # print("文件向量的長度：", len(d_vec))
  print("---------------------------------")

=== Bi-encoder ===
問題向量的長度： 1024
bi=0.880  勤続3年未満の場合、退職金は支給されない。
---------------------------------
bi=0.841  勤続3年未満の場合、退職金は支給される。
---------------------------------
bi=0.521  通勤手当は月額3万円を上限として支給する。
---------------------------------


In [27]:
# 5. Cross-encoder：問題和文件放在一起，直接給分數

print("=== cross-encoded ===")
for doc in docs:
  pair = (query, doc)
  score = cross.predict(pair)
  print(f"cross={score:.3f}  {doc}")


=== cross-encoded ===
cross=0.999  勤続3年未満の場合、退職金は支給されない。
cross=0.998  勤続3年未満の場合、退職金は支給される。
cross=0.000  通勤手当は月額3万円を上限として支給する。
